# SRC-001 Initial Inventory

## Purpose

Characterize the physical and structural organization of SRC-001 before
performing content-level data analysis.

This notebook inventories files, directories, formats, sizes, and basic
structural properties required to determine how the source should be analyzed.

## Out of scope

This notebook does not:

- evaluate Root Cause Analysis quality;
- classify incidents or anomalies;
- preprocess or transform the source;
- generate embeddings;
- build a RAG pipeline;
- train machine-learning models;
- perform semantic analysis of the corpus.

## 1. Environment

Verify the execution environment used for the audit to ensure that the
notebook is running with the expected project virtual environment.

In [1]:
import sys
from collections import defaultdict
from src.data.inventory import DataInventory

import pandas as pd

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("pandas:", pd.__version__)

Python: 3.12.13 (main, Apr 25 2026, 18:10:19) [Clang 21.0.0 (clang-2100.0.123.102)]
Python executable: /Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/.venv/bin/python
pandas: 3.0.6


## 2. Source Location

Locate the project root and the raw SRC-001 directory without depending on
machine-specific absolute paths.

In [2]:
SOURCE_ID = "SRC-001"

inventory = DataInventory(SOURCE_ID)

PROJECT_ROOT = inventory.project_root
RAW_DATA_PATH = inventory.raw_data_path
FILE_EXTENSIONS = inventory.discover_file_extensions()

In [3]:
print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("File extensions:", FILE_EXTENSIONS)

Project root: /Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform
Raw data path: /Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd
File extensions: {'.md', '.txt', '.csv', '.pcap', '.db'}


## 3. File Inventory

Create an inventory of every file contained in SRC-001.

Only filesystem metadata is collected at this stage. File contents are not
loaded or interpreted.

In [4]:
raw_files = sorted(
    path
    for path in RAW_DATA_PATH.iterdir()
    if path.is_file()
)

raw_files

[PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/Lower_Layer_Data.db'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/Network_Dataset.csv'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/README.md'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/benign.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/benign.txt'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/ddos_icmp_hping3.pcap'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/ddos_icmp_hping3.txt'),
 PosixPath('/Users/emil

In [5]:
inventory = pd.DataFrame(
    [
        {
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "size_mb": path.stat().st_size / (1024 ** 2),
        }
        for path in raw_files
    ]
)

inventory

,file_name,extension,size_bytes,size_mb
0,Lower_Layer_Data.db,.db,5402624,5.152344
1,Network_Dataset.csv,.csv,227184406,216.659933
2,README.md,.md,292,0.000278
3,benign.pcap,.pcap,71108907,67.814738
4,benign.txt,.txt,844614,0.805487
5,ddos_icmp_hping3.pcap,.pcap,224211674,213.824915
6,ddos_icmp_hping3.txt,.txt,535875,0.511050
7,ftp_hydra.pcap,.pcap,403521743,384.828322
8,ftp_hydra.txt,.txt,217793,0.207704
9,icmp_nping.pcap,.pcap,505113905,481.714158


## 4. File Type Summary

Summarize the physical formats available in SRC-001.

In [6]:
file_type_summary = (
    inventory
    .groupby("extension")
    .agg(
        file_count=("file_name", "count"),
        total_size_mb=("size_mb", "sum"),
    )
    .sort_values("total_size_mb", ascending=False)
)

file_type_summary

,file_count,total_size_mb
extension,,
.pcap,6,2115.206081
.csv,1,216.659933
.db,1,5.152344
.txt,6,2.354179
.md,1,0.000278


In [7]:
total_files = len(inventory)
total_size_mb = inventory["size_mb"].sum()

print(f"Total files: {total_files}")
print(f"Total size: {total_size_mb:.2f} MB")

Total files: 15
Total size: 2339.37 MB


## 5. File Size Summary

Inspect the size of each raw file before deciding how subsequent notebooks
should read and process each format.

In [8]:
file_size_summary = (
    inventory[
        ["file_name", "extension", "size_mb"]
    ]
    .sort_values("size_mb", ascending=False)
    .reset_index(drop=True)
)

file_size_summary

,file_name,extension,size_mb
0,portscan_masscan.pcap,.pcap,833.011580
1,icmp_nping.pcap,.pcap,481.714158
2,ftp_hydra.pcap,.pcap,384.828322
3,Network_Dataset.csv,.csv,216.659933
4,ddos_icmp_hping3.pcap,.pcap,213.824915
5,web_xss.pcap,.pcap,134.012367
6,benign.pcap,.pcap,67.814738
7,Lower_Layer_Data.db,.db,5.152344
8,benign.txt,.txt,0.805487
9,ddos_icmp_hping3.txt,.txt,0.511050


In [9]:
large_files = file_size_summary[
    file_size_summary["size_mb"] >= 100
]

large_files

,file_name,extension,size_mb
0,portscan_masscan.pcap,.pcap,833.011580
1,icmp_nping.pcap,.pcap,481.714158
2,ftp_hydra.pcap,.pcap,384.828322
3,Network_Dataset.csv,.csv,216.659933
4,ddos_icmp_hping3.pcap,.pcap,213.824915
5,web_xss.pcap,.pcap,134.012367


## 6. Scenario File Pairing

Verify whether scenario-level PCAP files have corresponding TXT files and
identify structurally incomplete scenario pairs.

In [10]:
available_extensions = sorted(
    {
        path.suffix.lower()
        for path in raw_files
        if path.suffix
    }
)

available_extensions

['.csv', '.db', '.md', '.pcap', '.txt']

In [11]:
files_by_stem = defaultdict(set)

for path in raw_files:
    files_by_stem[path.stem].add(path.suffix.lower())

files_by_stem

defaultdict(set,
            {'Lower_Layer_Data': {'.db'},
             'Network_Dataset': {'.csv'},
             'README': {'.md'},
             'benign': {'.pcap', '.txt'},
             'ddos_icmp_hping3': {'.pcap', '.txt'},
             'ftp_hydra': {'.pcap', '.txt'},
             'icmp_nping': {'.pcap', '.txt'},
             'portscan_masscan': {'.pcap', '.txt'},
             'web_xss': {'.pcap', '.txt'}})

In [12]:
scenario_extensions = set().union(
    *[
        extensions
        for extensions in files_by_stem.values()
        if len(extensions) > 1
    ]
)

scenario_extensions

{'.pcap', '.txt'}

In [13]:
scenario_names = sorted(
    stem
    for stem, extensions in files_by_stem.items()
    if extensions & scenario_extensions
)

scenario_pairs = pd.DataFrame(
    [
        {
            "scenario": stem,
            **{
                f"has_{extension.lstrip('.')}":
                    extension in files_by_stem[stem]
                for extension in sorted(scenario_extensions)
            },
        }
        for stem in scenario_names
    ]
)

scenario_pairs

,scenario,has_pcap,has_txt
0,benign,True,True
1,ddos_icmp_hping3,True,True
2,ftp_hydra,True,True
3,icmp_nping,True,True
4,portscan_masscan,True,True
5,web_xss,True,True


## 7. Basic Integrity Checks

Perform lightweight filesystem-level checks before content-specific analysis.
These checks do not validate the internal correctness of PCAP, CSV, SQLite,
or telemetry records.

In [14]:
empty_files = inventory[
    inventory["size_bytes"] == 0
]

empty_files

,file_name,extension,size_bytes,size_mb


In [15]:
duplicate_names = inventory[
    inventory["file_name"].duplicated(keep=False)
]

duplicate_names

,file_name,extension,size_bytes,size_mb


In [16]:
files_without_extension = inventory[
    inventory["extension"] == ""
]

files_without_extension

,file_name,extension,size_bytes,size_mb


In [17]:
print(f"Empty files: {len(empty_files)}")
print(f"Duplicate file names: {len(duplicate_names)}")
print(f"Files without extension: {len(files_without_extension)}")

Empty files: 0
Duplicate file names: 0
Files without extension: 0


## 8. Initial Findings

The initial filesystem-level audit of SRC-001 identified:

- 15 raw source files with a total size of approximately 2339.37 MB;
- 6 PCAP files, representing the largest portion of the source at approximately 2115.21 MB;
- 6 TXT telemetry files with a combined size of approximately 2.35 MB;
- 1 consolidated network dataset in CSV format of approximately 216.66 MB;
- 1 lower-layer SQLite database of approximately 5.15 MB;
- 1 Markdown documentation file.

Six scenario-level file pairs were identified:

- `benign`
- `ddos_icmp_hping3`
- `ftp_hydra`
- `icmp_nping`
- `portscan_masscan`
- `web_xss`

Each identified scenario contains both a PCAP and a TXT file.

The basic integrity checks detected:

- no empty files;
- no duplicate file names;
- no files without an extension.

The source is therefore structurally suitable for proceeding to content-level
inspection.

No conclusions are made at this stage regarding data quality, temporal
alignment, label correctness, scenario coverage, or suitability for Root Cause
Analysis.